<a href="https://colab.research.google.com/github/ahmdarwish/LLM-Agentic-Legal-Information-Retrieval/blob/main/code_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import kagglehub
kagglehub.login()
path = kagglehub.competition_download('llm-agentic-legal-information-retrieval')

100%|██████████| 753M/753M [00:30<00:00, 25.9MB/s]

Extracting files...


Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
import pandas as pd
import os
from sentence_transformers import SentenceTransformer, util
import torch


base_path = path + '/'

print(f"Loading data from: {base_path}")

 2. Load Data
train = pd.read_csv(os.path.join(base_path, 'train.csv'))
test = pd.read_csv(os.path.join(base_path, 'test.csv'))
laws = pd.read_csv(os.path.join(base_path, 'laws_de.csv'))

courts = pd.read_csv(os.path.join(base_path, 'court_considerations.csv'), nrows=50000)

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

corpus_df = pd.concat([
    laws[['citation', 'text']].rename(columns={'text': 'content'}),
    courts[['citation', 'text']].rename(columns={'text': 'content'})
]).reset_index(drop=True)

corpus_embeddings = model.encode(corpus_df['content'][:15000].tolist(), convert_to_tensor=True)

def get_citation(query):
    query_emb = model.encode(query, convert_to_tensor=True)
    hits = util.semantic_search(query_emb, corpus_embeddings, top_k=1)
    return corpus_df.iloc[hits[0][0]['corpus_id']]['citation']

print("Processing test queries...")
test['predicted_citations'] = test['query'].apply(get_citation)

test[['query_id', 'predicted_citations']].to_csv('submission.csv', index=False)
print("✅ submission.csv created!")

Loading data from: /root/.cache/kagglehub/competitions/llm-agentic-legal-information-retrieval/


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Processing test queries...
✅ submission.csv created!


In [4]:
print(laws[['citation', 'text']].head())

test['query_length'] = test['query'].apply(lambda x: len(x.split()))
print(f"Average query length: {test['query_length'].mean():.2f} words")

            citation                                               text
0         Art. 1 112  Die Einwohnergemeinde Bern tritt der Schweizer...
1         Art. 2 112  Die Einwohnergemeinde Bern wird ferner der Sch...
2  Art. 3 Abs. 1 112  1 Falls die Schweizerische Eidgenossenschaft z...
3  Art. 3 Abs. 2 112  2 Durch Anlage des neuen Verwaltungsgebäudes a...
4  Art. 4 Abs. 1 112  1 Die Einwohnergemeinde Bern übernimmt im fern...
Average query length: 218.53 words
